# Goals
Be able to predict individuals who will have a stroke based on health records.
Which factor correlates the strongest with stroke risk?


In [42]:
# data loading and library imports
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
import mylib

from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

output_dir = '../../../public/graphs/stroke'
os.makedirs(output_dir, exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("\nDATA LOADING")

df = pd.read_csv('../../data/healthcare-dataset-stroke-data.csv')
df


DATA LOADING


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


# I. Data Cleaning and Preparation

In [36]:
# Information on the dimensions, column titles, and data types (of the columns)
print(f"Dataset Shape: {df.shape}")
print(f"Dataset Dimesions: {df.columns}")
print(f"Dataset Types: {df.dtypes}")

# 1. Missing Value Analysis
mylib.missing_value_analysis(df)
# only bmi seems to have null/missing values. Let's check its distribution. 
skew_value = df['bmi'].skew()
print("BMI Skewness:", skew_value)
if(skew_value >= .5):
    print("\nModerate-High Skew; Median Imputed Preferred; Amount of Data Affected:", round(df['bmi'].isnull().sum()/(df.shape[0]*.01),4),"%")

df['bmi'] = df['bmi'].fillna(df['bmi'].median())
df

Dataset Shape: (5110, 12)
Dataset Dimesions: Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')
Dataset Types: id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

1. Missing Value Analysis:

     Missing_Count  Missing_Percentage
bmi            201            3.933464
BMI Skewness: 1.0553402052962912

Moderate-High Skew; Median Imputed Preferred; Amount of Data Affected: 3.9335 %


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,28.1,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [43]:
# I want to change all categorical to category not object. addictionally, I would like to create ordinal varables from some continuous variabler such as age.

# 2. Data Type Validation and Conversion
print("\n2. Data Type Validation:")
print(df.dtypes)

## Converting categorical columns to category type for mem/speed efficiency (usually object takes more memory)
categorical_cols = mylib.get_categorical_cols(df)
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("\nUpdated data type(s):")
print(df.dtypes)



2. Data Type Validation:
id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object


AttributeError: module 'mylib' has no attribute 'get_categorical_cols'

In [ ]:
# 3. Outlier Detection
print("\n3. Outlier Detection:")

def detect_outliers(df, column):
    """Detect outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = (Q3-Q1)*1.5
    lower_bound = Q1-IQR
    upper_bound = Q3+IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column]  > upper_bound)] # returns rows that are an outlier of the column
    return outliers, upper_bound, lower_bound # want to display the upper and lower bound for the post

numeric_columns = ['Daily_Minutes_Spent', 'Posts_Per_Day', 'Likes_Per_Day', 'Follows_Per_Day']

outlier_summary = {}
for col in numeric_columns:
    outliers, upper, lower = detect_outliers(df, col)
    outlier_summary[col] = {
        'outlier_count': len(outliers),
        'outlier_percentage': (len(outliers) / len(df)) * 100,
        'lower_bound': lower,
        'upper_bound': upper
    }
print("Outlier Summary:")
for col, info in outlier_summary.items():
    print(f"  {col}: {info['outlier_count']} outliers ({info['outlier_percentage']:.1f}%)")